# Agentic keyword-assignment workflow

This workflow uses the model as the planner:
1. retrieve relevant examples
2. ask the model for candidate keywords
3. map each candidate to the canonical vocabulary
4. return the final mapped result as a DataFrame

In [ ]:
from typing import Any

import pandas as pd
import weaviate
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.capabilities import Thinking

from src.retrieve import ExampleRetriever
from src.mapping import Mapping

# --- model setup for vLLM OpenAI-compatible API ---
model = OpenAIChatModel(
    "Qwen/Qwen3-4B",
    provider=OpenAIProvider(
        base_url="http://localhost:9513/v1",
        api_key="unused",
    ),
)

class KeywordAssignmentResult(BaseModel):
    Normierte_Schlagworte: list[str] = Field(..., description="Finale normierte Schlagworte aus dem kanonischen Vokabular")
    Freie_Schlagworte: list[str] = Field(..., description="Vom Modell vorgeschlagene freie Schlagworte")
    Normierte_Schlagworte_mit_id: list[dict[str, Any]] = Field(default_factory=list, description="Normierte Schlagworte zusammen mit ihren IDs aus dem kanonischen Vokabular")

agent = Agent(
    model,
    output_type=KeywordAssignmentResult,
    system_prompt=(
        "Du bist ein Assistent für die Sacherschließung. "
        "Verwende retrieve(), um relevante Beispiele abzurufen, und schlage anschließend "
        "Kandidatenschlagwörter für den Eingabetext vor. "
        "Verwende dann map(), um jedes Kandidatenschlagwort auf das kanonische "
        "Schlagwortvokabular abzubilden. "
        "Prüfe, ob die abgebildeten Schlagwörter weiterhin zum Eingabetext passen, "
        "und schließe irrelevante oder falsche Zuordnungen aus. "
        "Gib die finalen abgebildeten Labels als JSON zurück."
    ),
    capabilities=[Thinking()],
)

@agent.tool
def retrieve(ctx: RunContext[None], text: str, n_examples: int = 5) -> list[dict]:
    df = pd.DataFrame(
        {
            "text": [text],
            "doc_id": [1],
            "label_ids": [[1]],
            "label_texts": [[text]],
        }
    )
    retriever = ExampleRetriever(
        input_text_data=df,
        output_file=None,
        n_examples=n_examples,
        collection_name="title_train",
        host="8090",
        debug=False,
    )
    return retriever.retrieve_examples().to_dict(orient="records")

@agent.tool
def map(ctx: RunContext[None], candidate: str) -> dict:
    client = weaviate.connect_to_local(port=8087)
    try:
        mapper = Mapping(
            hyperparameters={
                "host": "8090",
                "alpha": 0.7,
                "use_phrase": False,
                "search": "hybrid",
            },
            collection_name="ki_fsprompt_vocab",
            phrase=None,
            debug=False,
            db_connection=client,
        )
        result = mapper.query_vector_database(candidate=candidate)
        item = next(iter(result.values()))
        return {
            "candidate": candidate,
            **item,
        }
    finally:
        client.close()



In [ ]:
text = "Serumprobe bei Patienten mit Verdacht auf Infektion"


result = await agent.run(text)

print("MESSAGES")
for m in result.all_messages():
    print(m)

print("OUTPUT")
print(result.output)